# Hotel Booking Cancellation Prediction

**Tabular Classification Project**

## 2 · Project Overview

Predict whether a hotel booking will be **canceled** based on booking details: lead time, deposit type, previous cancellations, special requests, and guest demographics. Synthetic dataset with ~10,000 rows and ~37% cancellation rate.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given booking details (lead time, room type, deposit, guest history, special requests), predict whether the booking will be canceled (is_canceled = 1) or honored (is_canceled = 0).

## 5 · Why This Project Matters

- Hotel cancellations cost the industry **billions annually** in lost revenue.
- Predicting cancellations enables **overbooking strategies** and dynamic pricing.
- Lead time and deposit type are strong behavioral signals.
- This teaches feature engineering from booking metadata.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~10,000 |
| Features | 12 (lead_time, arrival_month, stays_weekend, stays_weeknight, adults, children, meal, country, market_segment, deposit_type, previous_cancellations, total_special_requests) |
| Target | `is_canceled` (binary: 0=not canceled, 1=canceled) |
| Class balance | ~63% not canceled, ~37% canceled |
| Missing values | None |

## 7 · Dataset Source and License Notes

**Source**: Hotel Booking Demand dataset (Antonio, Almeida & Nunes, 2019).
**License**: Public / educational.
**Note**: Synthetic reproduction of hotel booking patterns.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "is_canceled"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

In [ ]:
np.random.seed(SEED)
n = 10000

lead_time = np.random.exponential(100, n).clip(0, 700).astype(int)
arrival_month = np.random.randint(1, 13, n)
stays_weekend = np.random.poisson(0.9, n).clip(0, 6)
stays_weeknight = np.random.poisson(2.5, n).clip(0, 15)
adults = np.random.choice([1, 2, 3], n, p=[0.25, 0.65, 0.1])
children = np.random.choice([0, 1, 2, 3], n, p=[0.7, 0.2, 0.08, 0.02])
meal = np.random.choice(['BB', 'HB', 'SC', 'FB'], n, p=[0.5, 0.25, 0.2, 0.05])
country = np.random.choice(['PRT', 'GBR', 'FRA', 'ESP', 'DEU', 'Other'], n,
                            p=[0.3, 0.15, 0.12, 0.1, 0.08, 0.25])
market_segment = np.random.choice(['Online_TA', 'Offline_TA', 'Direct', 'Corporate', 'Groups'],
                                   n, p=[0.45, 0.2, 0.15, 0.12, 0.08])
deposit_type = np.random.choice(['No_Deposit', 'Non_Refund', 'Refundable'], n, p=[0.75, 0.2, 0.05])
previous_cancellations = np.random.poisson(0.1, n).clip(0, 10)
total_special_requests = np.random.poisson(0.6, n).clip(0, 5)

score = (
    0.005 * lead_time
    + 0.8 * (deposit_type == 'Non_Refund').astype(float)
    + 0.5 * previous_cancellations
    - 0.3 * total_special_requests
    - 0.15 * children
    + 0.1 * (market_segment == 'Online_TA').astype(float)
    + np.random.normal(0, 0.8, n)
)
is_canceled = (score > 0.7).astype(int)

df = pd.DataFrame({
    'lead_time': lead_time, 'arrival_month': arrival_month,
    'stays_weekend': stays_weekend, 'stays_weeknight': stays_weeknight,
    'adults': adults, 'children': children, 'meal': meal,
    'country': country, 'market_segment': market_segment,
    'deposit_type': deposit_type, 'previous_cancellations': previous_cancellations,
    'total_special_requests': total_special_requests, 'is_canceled': is_canceled,
})
print(f"Dataset shape: {df.shape}")
print(f"Cancellation rate: {df['is_canceled'].mean():.2%}")
df.head()

## 12 · Data Validation Checks

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

## 13 · Exploratory Data Analysis

In [ ]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(num_cols[:6]):
    ax = axes[i // 3, i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(f"{col} by Cancellation")
plt.suptitle("")
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(['deposit_type', 'market_segment', 'meal']):
    ct = pd.crosstab(df[col], df[TARGET], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=axes[i], color=['steelblue', 'coral'])
    axes[i].set_title(f"Cancellation by {col}")
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()
print(f"Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}")

## 14 · Target Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title("Booking Cancellation Distribution")
axes[0].set_xticklabels(['Not Canceled (0)', 'Canceled (1)'], rotation=0)
df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue', 'coral'])
axes[1].set_title("Class Proportions"); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts()}")

## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    df[cat_cols] = oe.fit_transform(df[cat_cols])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [ ]:
X_train = X_train.copy(); X_test = X_test.copy()
X_train['total_stay'] = X_train['stays_weekend'] + X_train['stays_weeknight']
X_test['total_stay'] = X_test['stays_weekend'] + X_test['stays_weeknight']
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean; X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

## 18 · Baseline Model

Logistic Regression as baseline.

In [ ]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
n_classes = len(np.unique(y_train))
if n_classes == 2:
    y_prob_bl = baseline.predict_proba(X_test)[:, 1]
else:
    y_prob_bl = None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

In [ ]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [ ]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

## 22 · Top 3 Model Selection

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

## 25 · Interpretation and Business Insight

- **Lead time** is the strongest predictor — longer lead times mean higher cancellation.
- **Deposit type** (Non-Refundable) strongly predicts cancellation behavior.
- **Previous cancellations** are a strong repeat-behavior signal.
- **Special requests** reduce cancellation — invested guests honor bookings.
- **Children** in the booking slightly reduce cancellation risk.

## 26 · Limitations

1. Synthetic data — real hotel data has seasonal patterns and external factors.
2. No pricing information (room rate, discounts).
3. No booking channel details beyond market segment.
4. No temporal sequence of booking modifications.
5. Country encoding loses geographic nuance.

## 27 · How to Improve This Project

1. Add room rate and revenue per booking for cost-sensitive modeling.
2. Engineer lead-time buckets and seasonal features.
3. Include booking modification history.
4. Build a revenue-optimization model (overbooking vs cancellation cost).
5. Apply survival analysis for time-to-cancellation.

## 28 · Production Considerations

- Real-time cancellation risk scoring at booking time.
- Integration with revenue management systems.
- Overbooking policy optimization.
- Regular retraining as booking patterns change post-COVID.

## 29 · Common Mistakes

1. Not accounting for deposit type's strong signal.
2. Treating all cancellations equally (revenue impact varies).
3. Ignoring temporal patterns in booking lead time.
4. Using accuracy when cost-sensitive evaluation matters.
5. Not validating on different hotel types separately.

## 30 · Mini Challenge / Exercises

1. Remove lead_time and retrain — how much does F1 drop?
2. Build separate models for resort vs city hotels.
3. Create a cost matrix: what's the cost of false positive vs false negative?
4. Engineer a 'booking_changes' feature from stays and special requests.

## 31 · Final Summary / Key Takeaways

1. Hotel cancellation prediction is a high-value hospitality ML problem.
2. Lead time and deposit type are dominant predictors.
3. Previous cancellation history is a strong behavioral signal.
4. Special requests indicate guest commitment.
5. Production deployment needs cost-sensitive evaluation, not just accuracy.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))